# Day 5 — Production support scenarios (Jupyter + CMD)

**Prerequisites:** Days 1 to 4 complete.

Today is different from the previous four days. You are not learning new commands — you already have all of them. You are learning to use them **under the conditions of a real support ticket**: incomplete information, someone waiting for an answer, and pressure to do something visible quickly.

Each scenario starts with a ticket written the way tickets actually arrive — vague, slightly wrong, and confident about the cause. Your job is to work out what is actually happening before you change anything.

### The playbook — use it on all five scenarios

| Step | What you do | Why it is in this order |
|------|-------------|-------------------------|
| **1. Symptom** | Write down what the ticket claims, in the reporter's words | Separates what was *reported* from what you *find* |
| **2. Scope** | One topic, one consumer group, or the whole cluster? | Decides urgency and who needs telling |
| **3. Evidence** | Application logs, `--describe`, CloudWatch — all for the same time window | Without a shared time window you are comparing unrelated facts |
| **4. One fix** | Make a single change | Two changes at once means you never learn which one worked |
| **5. Validate** | Send a named test message and confirm `LAG` reaches 0 | "It's running" is not proof. A message arriving is. |

### Why "one fix" is a hard rule

The pressure in a live incident pushes the other way: restart the consumer *and* reset the offsets *and* bounce the app, because one of them will probably work. It often does — and you have then learned nothing, cannot write an honest root cause, and will meet the same incident again next week.

One change, then validate. If it did not work, you have ruled something out, which is also progress.

### Course map

| Scenario | Focus |
|----------|-------|
| **1** | Application logs, live path testing, broker metrics, restoring processing |
| **2** | Broker availability, DNS and network path, resource constraints |
| **3** | Missing messages, offset validation, reset and replay |
| **4** | Authentication, authorization, listener and configuration validation |
| **5** | A full end-to-end incident, plus the daily operations checklist |

Every code cell runs CMD commands from [commands.md](commands.md). Theory: [notes.md](notes.md).

### Ground rules on the shared cluster

| Rule | Why |
|------|-----|
| Use **your own** ids only — `orders-userN`, `cg-userN-support`, `acl-lab-userN` | Another seat is mid-scenario on theirs |
| Never reset offsets as your **first** action | It is a data decision, not a diagnostic step |
| Never restart or resize the MSK cluster | That is an infrastructure change with a real blast radius; you escalate instead |


## Setup — load the lab session

Every cell begins with `call ..\scripts\jupyter-lab-session.bat`, which loads your ids and endpoints into that cell's CMD process.

### The two kinds of evidence you will use today, and how to tell them apart

This distinction is the single most important habit of the day, so it is worth being explicit about:

| Evidence | What it is | What it can prove |
|----------|-----------|-------------------|
| **Sample application logs** — the `type ..\day-XX\samples\...` cells | Recorded log files from an incident, with historical timestamps | What the *application* experienced during that incident |
| **Live commands and CloudWatch** — everything you run today | Your own cluster, right now | Whether the path is healthy *at this moment* |

The sample logs are dated in the past, so there is nothing in CloudWatch for their exact timestamps. That is not a gap in the lab — it is the normal situation when a ticket arrives with logs attached from earlier that day and you need to establish whether the problem is still happening.

**When you write your findings, label every fact as `[log]` or `[live]`.** It takes three characters and it is the difference between a report someone can act on and a report they have to come back and question.

**On CloudWatch:** use the AWS Console with a **Last 1 hour** time range, exactly as on Day 3. That covers the live commands you run today, which is what you actually need to know about.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo LOGIN=%LOGIN% TOPIC=%TOPIC% GROUP=%GROUP% ACL_TOPIC=%ACL_TOPIC%
echo CLUSTER_NAME=%CLUSTER_NAME% REGION=%REGION%


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo LOGIN=%LOGIN% TOPIC=%TOPIC% GROUP=%GROUP% ACL_TOPIC=%ACL_TOPIC%
LOGIN=user15 TOPIC=orders-user15 GROUP=cg-user15-support ACL_TOPIC=acl-lab-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo BOOTSTRAP=%BOOTSTRAP%
BOOTSTRAP=b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com:9196

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo BOOTSTRAP_IAM=%BOOTSTRAP_IAM%
BOOTSTRAP_IAM=b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com:9198

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

c:-Trainings-Confirmed-08-26-kafka\GH\day-05>echo CLUSTER_NAME=%CLUSTER_NAME% REGION=%REGION%
CLUSTER_NAME=msk-kafka-class REGION=ap-south-1

c:-Trainings-Confirmed-08-26-kafka\GH\day-05>

---
## Scenario 1 — Deployed applications

### The ticket

> **INC-4471** — raised by Order Fulfilment, 09:14
>
> *"The orders dashboard stopped updating some time this morning. Customers are ringing us. The Kafka team needs to fix Kafka."*
>
> *Attached: two log files from the application host.*

### What the ticket tells you, and what it does not

| The ticket says | What you actually know |
|-----------------|------------------------|
| "stopped updating some time this morning" | No usable time. You need a first-error timestamp from the logs. |
| "The Kafka team needs to fix Kafka" | An assumption, not a finding. Nobody has yet shown the cluster is involved. |
| Two attached log files | Your only hard evidence so far |

That last row of the middle column matters. Reporters routinely name the cause in the ticket title, and they are often wrong — not because they are careless, but because from where they sit, "data stopped arriving from Kafka" and "Kafka is broken" look identical.

### The tempting move, and why to resist it

The fastest visible action is to restart the consumer, or reset the offsets so the dashboard refills. Either might appear to work.

Neither tells you what happened. If the cause is still present, you will be back here within the hour, with a less patient caller and no more information than you have now.

### Your first move: establish a timeline and a blast radius

Three questions, in this order:

1. **When did it start?** From the application logs.
2. **Is the cluster reachable at all right now?** From your own client. This is the question that decides whether the ticket belongs to your team.
3. **Were the brokers under stress?** From CloudWatch.


### Step 1 — read the producer log

Read it in time order and look for these four things. You met all of them on Day 2:

| What to find | What it means |
|--------------|---------------|
| **`ProducerConfig`** with `acks=all` | Every write is waiting for all in-sync replicas. Sensitive to replica problems. |
| **`NetworkClient` WARN** | The client could not open a connection to a broker at all |
| **`NOT_LEADER_OR_FOLLOWER`** | The partition leader moved; the client refreshes metadata and retries. A few of these are normal. |
| **`TimeoutException`** | The message ran out of its `delivery.timeout.ms` budget and was discarded |

**Write down the first and last error timestamps.** That is your incident window, and every other piece of evidence today gets compared against it.

**The key question this log answers:** did the producer ever reach a broker? A `NetworkClient` warning followed by timeouts says the messages never landed. That points at the network path or the app host, not at broker capacity.


### Step 2 — read the consumer log

The same log file tells a second story, from the reading side:

| What to find | What it means |
|--------------|---------------|
| **Joined group**, partitions assigned | The consumer was healthy at this point |
| **`poll timeout has expired`** | The application stopped calling `poll()` in time — processing was slow or blocked |
| **`CommitFailedException`** | The consumer was removed from the group, so its progress was never saved |
| **`DisconnectException`** | The consumer loop stopped |

That sequence is a **slow or stuck consumer**, not a broken broker. The broker did exactly what it is designed to do: a member stopped doing useful work, so the group took its partitions away and gave them to somebody else.

**The consequence to note for the ticket:** because the commit failed, messages between the last successful commit and the failure will be **processed again** when the application restarts. Whoever owns the downstream system needs to know that.


In [ ]:
%%cmd
type ..\day-02\samples\producer-error.log


2026-08-20 10:14:02,101 INFO  org.apache.kafka.clients.producer.ProducerConfig - ProducerConfig values:
	acks = all
	bootstrap.servers = [b-1-public.example.kafka.amazonaws.com:9196]
	delivery.timeout.ms = 120000
	enable.idempotence = false
	retries = 5
	retry.backoff.ms = 100
	request.timeout.ms = 30000

2026-08-20 10:14:03,220 INFO  org.apache.kafka.clients.Metadata - [Producer clientId=order-producer-lab] Cluster ID: msk-lab-demo

2026-08-20 10:18:41,004 WARN  org.apache.kafka.clients.NetworkClient - [Producer clientId=order-producer-lab] Connection to node 1 (b-1-public.example.kafka.amazonaws.com/203.0.113.10:9196) could not be established. Broker may not be available.

2026-08-20 10:18:41,188 INFO  org.apache.kafka.clients.NetworkClient - [Producer clientId=order-producer-lab] Give up sending metadata request since no node is available

2026-08-20 10:18:46,201 WARN  org.apache.kafka.clients.producer.internals.Sender - [Producer clientId=order-producer-lab] Got error produce respo

In [ ]:
%%cmd
type ..\day-02\samples\consumer-error.log


2026-08-20 10:19:10,044 INFO  org.apache.kafka.clients.consumer.ConsumerConfig - ConsumerConfig values:
	bootstrap.servers = [b-1-public.example.kafka.amazonaws.com:9196]
	group.id = cg-lab-support
	enable.auto.commit = true
	max.poll.interval.ms = 300000
	session.timeout.ms = 45000
	auto.offset.reset = earliest

2026-08-20 10:19:11,201 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=cg-lab-support] Discovered group coordinator b-2-public.example.kafka.amazonaws.com:9196 (id: 2147483646 rack: null)

2026-08-20 10:19:11,388 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=cg-lab-support] Successfully joined group with generation Generation{generationId=7, memberId='order-consumer-lab-1', protocol='range'}

2026-08-20 10:19:11,401 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=c

### Step 3 — prove whether the path works from your own machine

This is the step that most often changes the direction of a ticket, and it takes about a minute.

The logs tell you what the *application host* experienced. They say nothing about whether the cluster is reachable **now**, from **somewhere else**. Your lab PC is that somewhere else.

Run these four cells in order:

| Cell | What it proves if it succeeds |
|------|-------------------------------|
| `--list` | Network path, TLS, and SCRAM authentication all work from your machine |
| `--describe` on the group | The current lag, before you change anything — your baseline |
| `produce.bat s1-probe` | A brand new message can be written to the topic right now |
| `consume.bat` | That same message can be read back — the full path is working |

**If all four succeed**, the cluster is healthy and the ticket is not a Kafka outage. The evidence points at the application host or its configuration, and that changes who owns the next step.

**`s1-probe` is just a message whose text you chose** so you can recognise it in the output. Seeing that exact word come out proves a message you sent *seconds ago* travelled the whole path — much stronger evidence than a process being up.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


acl-lab-user15
orders-demo
orders-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === Lag BEFORE the probe ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat s1-probe


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\produce.bat recover-probe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\consume.bat --max-messages 10 --timeout-ms 25000


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\consume.bat --from-beginning --max-messages 20 --timeout-ms 20000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


order-1001
order-1002


Processed a total of 2 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === Lag AFTER consuming ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


### Step 4 — check the broker side in CloudWatch

Open the AWS Console exactly as you did on Day 3:

1. **CloudWatch** → **Metrics** → **All metrics** → **`AWS/Kafka`**
2. Dimension group **Broker ID, Cluster Name**
3. Select **`CpuIdle`** and **`KafkaDataLogsDiskUsed`** for all three brokers
4. Statistic **Average**, Period **1 minute**, time range **Last 1 hour**

Record what you find:

| Metric | Value now | Healthy? |
|--------|-----------|----------|
| `CpuIdle`, lowest of the three brokers | | |
| `KafkaDataLogsDiskUsed`, highest of the three | | |

**Why you are looking at now, rather than the log's timestamp.** The attached logs are from an earlier incident, so CloudWatch has nothing for those exact minutes. What matters for this ticket is whether the cluster is healthy **while your own probe is succeeding**. Healthy metrics plus a successful probe is a complete, defensible answer to "is Kafka the problem?" — and it is the answer you can give in the next five minutes rather than the next hour.


### Diagnosis and close — Scenario 1

Put the evidence together:

| Evidence | Source | What it establishes |
|----------|--------|---------------------|
| Producer timeouts and `NetworkClient` warnings | `[log]` | The application host could not reach a broker during the incident |
| Poll timeout, then `CommitFailedException` | `[log]` | The consumer was stuck, was evicted from the group, and lost its committed progress |
| `--list`, probe produce and consume all succeed | `[live]` | The cluster, the network path, and authentication are all fine right now |
| `CpuIdle` high, disk steady | `[live]` | The brokers have plenty of headroom |

**Conclusion:** this is not a cluster outage. It is a consumer-side failure, on a healthy cluster.

**The one fix:** restore the consumer path — which is exactly what the probe-and-consume cells did. In production this is the application team restarting their consumer service, ideally after finding out why processing became slow enough to breach `max.poll.interval.ms`.

### The update you write on the ticket

> Investigated INC-4471. The MSK cluster is healthy: `--list`, a test publish, and a test consume all succeeded from an independent client, with `CpuIdle` above 90% and log disk steady on all three brokers.
>
> The attached logs show the consumer application breaching its poll interval, being removed from the consumer group, and failing to commit its offsets. Producer timeouts in the same window indicate the application host could not reach a broker while other clients could.
>
> Consumer processing has been restored and validated with a test message; lag has returned to 0. Reassigning to the application team to investigate why message processing slowed enough to breach the 5-minute poll interval. Please note that messages between the last successful commit and the failure will have been reprocessed.

### What you would deliberately not do, and why

| Not this | Why not |
|----------|---------|
| Reset the consumer offsets | Nothing suggested the saved position was wrong. A reset would have caused duplicate downstream processing for no reason. |
| Restart the MSK cluster | The cluster was demonstrably healthy. This would have caused a real outage while chasing a phantom one. |
| Accept "fix Kafka" as the scope | The evidence pointed elsewhere. Fixing the wrong layer wastes the outage window and guarantees a repeat. |


---
## Scenario 2 — Infrastructure and performance

### The ticket

> **INC-4488** — raised by the Payments platform team, 11:40
>
> *"Our service cannot connect to Kafka. It worked yesterday. We have not changed anything. Is the cluster down?"*
>
> *No logs attached.*

### What makes this ticket harder than the last one

There are no logs. All you have is a claim and an assumption. "We have not changed anything" is offered in good faith and is very frequently untrue — but it is also unfalsifiable, so arguing about it wastes time.

So you answer the only question that can actually be settled quickly: **is the cluster up, and is the path to it working?**

### The infrastructure checklist, in order

Each step is chosen to rule out one whole category of cause, cheapest first:

| # | Check | Rules out |
|---|-------|-----------|
| 1 | Is the MSK cluster **Active**, with the expected broker count? | A genuine cluster-level outage |
| 2 | Is the client using the **correct bootstrap endpoint** for where it runs? | The most common cause of "it cannot connect" |
| 3 | Does the broker hostname **resolve in DNS**? | A name resolution failure, which looks identical to a firewall block |
| 4 | Are the topic's partitions **healthy**, with full ISR? | Replication problems |
| 5 | Are the brokers **short of CPU or disk**? | Capacity as the underlying cause |

Notice that steps 2 and 3 are about the *client*, not the cluster. On this kind of ticket they are the answer far more often than anything cluster-side.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION% --query "ClusterInfo.{State:State,Brokers:NumberOfBrokerNodes}" --output table


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION% --query "ClusterInfo.{State:State,Brokers:NumberOfBrokerNodes}" --output table



aws: [ERROR]: An error occurred (AccessDeniedException) when calling the DescribeCluster operation: User: arn:aws:iam::410232017221:user/u1 is not authorized to perform: kafka:DescribeCluster on resource: arn:aws:kafka:ap-south-1:891377046325:cluster/msk-kafka-class/ad474b96-f594-495b-82cd-ad95d7c2c71c-4 because no resource-based policy allows the kafka:DescribeCluster action



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### Step 2 — confirm the endpoint matches where the client runs

An MSK cluster hands out **different bootstrap strings for different network positions**, and using the wrong one produces a timeout that looks exactly like an outage.

| Endpoint style | Port | Who it is for |
|----------------|------|---------------|
| `b-1-public...` | **9196** | Clients **outside** the VPC — including your lab PC |
| `b-1...` (no `-public`) | **9096** | Clients **inside** the VPC |

Your `%BOOTSTRAP%` must be a **`-public`** host on port **9196**. From outside the VPC the private `:9096` endpoint will simply time out, with no error explaining why.

**This is the first thing to ask on any "cannot connect" ticket:** where is the client running, and which endpoint is it configured with? A service that was moved from EC2 into a different network, or a configuration copied between environments, produces exactly this symptom — and exactly this "we changed nothing" report, because from the application team's point of view nothing in *their* code changed.

**If `describe-cluster` or `get-bootstrap-brokers` returns `AccessDenied`:** your AWS CLI credentials are for a different account from the cluster. Note it and continue — the Kafka path from your own client is what this scenario is really testing, and it does not depend on the AWS CLI.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka get-bootstrap-brokers --cluster-arn %CLUSTER_ARN% --region %REGION%


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>aws kafka get-bootstrap-brokers --cluster-arn %CLUSTER_ARN% --region %REGION%



aws: [ERROR]: An error occurred (AccessDeniedException) when calling the GetBootstrapBrokers operation: User: arn:aws:iam::410232017221:user/u1 is not authorized to perform: kafka:GetBootstrapBrokers on resource: arn:aws:kafka:ap-south-1:891377046325:cluster/msk-kafka-class/ad474b96-f594-495b-82cd-ad95d7c2c71c-4 because no resource-based policy allows the kafka:GetBootstrapBrokers action



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Step 3 — check DNS resolution

Before blaming a firewall, confirm the hostname resolves at all. A DNS failure and a blocked port produce the same user-visible symptom — "it just times out" — but they are fixed by different teams.

The cell below extracts the hostname from your `%BOOTSTRAP%` string and looks it up.

| Result | What it means | Who owns the next step |
|--------|---------------|------------------------|
| One or more **Address** lines returned | DNS is fine. If the connection still fails, suspect the port, the Security Group, or authentication. | Network team, with your evidence |
| **Non-existent domain** | The hostname is wrong, or the endpoint was regenerated after a cluster change | Whoever set the client configuration |
| **Request timed out** | The DNS resolver itself is unreachable from this host | Network team |

**What you do with the answer.** You do not change Security Group rules in this lab, and in most organisations you would not have permission to either. You collect three things — the bootstrap string, the nslookup result, and the exact Kafka error — and escalate with all three. An escalation carrying that evidence gets actioned; one saying "Kafka is not working" gets sent back for details.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
for /f "tokens=1 delims=:" %%a in ("%BOOTSTRAP%") do set BROKER_HOST=%%a
echo Resolving %BROKER_HOST%
nslookup %BROKER_HOST%


Resolving b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com
Server:  UnKnown
Address:  2401:9640:2f31::6

Name:    b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com
Address:  43.204.58.22

Non-authoritative answer:



### Step 4 — check partition health and consumer state

Two commands, two different questions.

**Topic `--describe`** answers *is the data healthy?* Look at the `Isr` column against `Replicas` on every partition:

| What you see | Meaning |
|--------------|---------|
| `Isr` matches `Replicas` | Fully replicated. Healthy. |
| `Isr` shorter than `Replicas` | **Under-replicated.** A replica has fallen behind or its broker is unavailable. With `acks=all`, produce can slow down or fail. |

**Group `--describe`** answers *is anything reading?* The column to check is `CONSUMER-ID`:

| What you see | Meaning |
|--------------|---------|
| A real consumer id per partition | Consumers are attached and working |
| `-` or `none` | **No active members.** Messages are accumulating and nothing is reading them. |

An empty `CONSUMER-ID` with growing `LAG` is the signature of the most common Kafka ticket there is, and it is a consumer problem every time.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo === BEFORE reset ===
=== BEFORE reset ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Step 5 — check broker resources, then restore the path

**In the AWS Console**, as on Day 3: `CpuIdle` and `KafkaDataLogsDiskUsed` per Broker ID, **Last 1 hour**.

| Finding | What it means | Your action |
|---------|---------------|-------------|
| `CpuIdle` low on **one** broker only | Leadership or partition **skew** | Recommend rebalancing leadership. Adding brokers will not help. |
| `CpuIdle` low on **all** brokers | Genuine capacity shortage | Escalate to infrastructure with the numbers |
| Disk above 80% | Retention is holding more than the volume can take | Escalate capacity, and review retention |
| Everything healthy | The cluster is not the problem | Say so clearly, and point at the evidence |

**You do not resize brokers in this lab**, and in production that is a change-managed operation, not an incident action. What you provide is the measurement that justifies it.

**Restoring the path.** If the group had no active members, start a consumer and publish a named test message — `s2-restore` — so you can prove the path works end to end. That is what the next cell does.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat s2-restore
call ..\scripts\consume.bat --max-messages 5 --timeout-ms 20000


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\produce.bat recover-probe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\consume.bat --from-beginning --max-messages 15 --timeout-ms 25000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


recover-probe


Processed a total of 1 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### Diagnosis and close — Scenario 2

| Evidence | Source | What it establishes |
|----------|--------|---------------------|
| Cluster **Active**, expected broker count | `[live]` | No cluster-level outage |
| `%BOOTSTRAP%` is a `-public` host on 9196 | `[live]` | The endpoint is correct for a client outside the VPC |
| `nslookup` returns addresses | `[live]` | DNS resolution is working |
| ISR matches Replicas on all partitions | `[live]` | Replication is healthy |
| `s2-restore` published and consumed | `[live]` | The full path works from an independent client |

**Conclusion:** the cluster is up and reachable. Since an independent client can connect while the reporting service cannot, the difference is on the client side — almost always the wrong bootstrap endpoint for where that service now runs, or a Security Group that does not permit its source.

### The update you write on the ticket

> Investigated INC-4488. The MSK cluster is Active with all 3 brokers, DNS resolves for the public bootstrap endpoint, and all partitions are fully replicated. I published and consumed a test message successfully from an independent client at 11:52, so the cluster and its network path are working.
>
> Because another client can connect, the failure is specific to the Payments service. Please confirm two things: which bootstrap string it is configured with, and which subnet it now runs in. A client outside the VPC must use the `-public` endpoint on port 9196; the private endpoint on 9096 times out with no explanatory error, which matches the symptom described.
>
> Attaching the cluster state, the bootstrap endpoint list, and the DNS resolution output.

### What you would deliberately not do, and why

| Not this | Why not |
|----------|---------|
| Change Security Group rules yourself | Cluster-wide blast radius, usually outside your permissions, and untraceable if it goes wrong. Escalate with evidence instead. |
| Resize or restart brokers because "it might be load" | You had no measurement supporting load. This is an expensive guess. |
| Accept "is the cluster down?" as the question | The useful question is "why can this one client not connect when others can?" |


---
## Scenario 3 — Missing messages and reprocessing

### The ticket

> **INC-4502** — raised by Customer Service, 14:05
>
> *"Order `order-7700` is not in the orders database. The customer has the confirmation email. Please replay the Kafka messages from this morning so it gets picked up."*

### Why this is the most dangerous ticket of the five

The reporter has done your thinking for you and asked for a specific action. The action is destructive, the request sounds reasonable, and saying yes takes ten seconds.

If you replay from this morning, **every** message from this morning is delivered again — not just `order-7700`. Depending on what the consumer does with them, that could mean duplicate database rows, duplicate emails to customers who did not ask for them, or duplicate payments.

So the first task is not to run anything. It is to find out whether a replay would even help.

### The four questions, before touching anything

| Question | How to check it | If this is the answer |
|----------|-----------------|------------------------|
| Was the message **ever produced**? | Producer logs — is there a successful publish for `order-7700`? | Nothing to recover. The problem is upstream of Kafka. |
| Has the group **already read past** it? | `--describe`: is `CURRENT-OFFSET` beyond that message? | Kafka delivered it. The message was lost inside the consumer. |
| Did it go to a **different partition**? | The message key decides the partition | It is there, just not where someone looked |
| Did **retention** delete it? | Compare the message's age with `retention.ms` | Replay cannot recover it. Say so immediately. |

Look at the second row again, because it is the one that resolves this ticket most often. If the group's committed offset is already past that message, Kafka handed it over successfully and the consumer failed to write it to the database. **Replaying will not fix a consumer bug** — it will run the same broken code over the same message and produce the same result, plus duplicates of everything else.

The scenario 5 log later today contains exactly this evidence: *"Missing order-7700 in downstream DB; last committed offset on partition 0 is 9100"*. That line says Kafka's part was done.


### Before you run anything — the preconditions

**1. Stop every consumer in `%GROUP%`.** A running consumer keeps committing its own position and will undo your reset. Kafka refuses to reset an active group, and getting past that check leaves you with a result nobody intended.

**2. Reset only your own `%GROUP%` and `%TOPIC%`.** Never another seat's group.

**3. Check retention first**, using the topic and configuration describe below. If the data is outside retention, stop here — there is nothing to replay and the honest answer is worth more than an attempt.

### What each step should show you

| Step | What you should see |
|------|---------------------|
| `--describe` on the topic, plus `kafka-configs` | Replication factor 3, ISR matching Replicas; retention either as an override or inherited from the broker default |
| `--describe` on the group | The `LAG` column, and `CURRENT-OFFSET` against `LOG-END-OFFSET` — note these before you change anything |
| `--dry-run` | **The plan only. Nothing has changed.** Read every partition and its proposed new offset. |
| `--execute` | The same plan applied. The committed offsets actually move. |
| `consume --from-beginning` | Old messages read again. **Downstream systems would see duplicates here.** |
| Final `--describe` | `CURRENT-OFFSET` has moved and `LAG` is back to 0 |


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe
kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Dynamic configs for topic orders-user15 are:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo === BEFORE reset ===
=== BEFORE reset ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --dry-run


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --dry-run


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



GROUP                          TOPIC                          PARTITION  NEW-OFFSET     cg-user15-support              orders-user15                  0          0              cg-user15-support              orders-user15                  1          0              cg-user15-support              orders-user15                  2          0              
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Read the dry-run before you execute

The dry-run prints one line per partition, giving the offset each one would be moved to. Nothing has changed yet.

**Three things to check in that output:**

1. **The group and topic names are yours.** A wrong `--group` on a shared cluster affects somebody else's work.
2. **Every partition is listed.** A reset that reaches only some partitions leaves the group in an inconsistent state.
3. **The new offsets are what you expected.** For `--to-earliest` they will be the oldest retained offset per partition — often 0, but not always, because retention may already have removed the earliest data.

**Paste this output into the ticket before you continue.** It is your record of what you intended, and the thing you will be grateful for when someone asks about it months later.

### A note on the reset target

This lab uses `--to-earliest` because it is easy to observe. In production, a request like this one usually wants **`--to-datetime`** instead:

`--reset-offsets --to-datetime 2026-09-01T09:00:00.000 --topic %TOPIC%`

That replays only from when the problem started, rather than everything on the topic. It maps the business statement — *"we lost data from about 09:00"* — onto precisely the messages involved, and it is very often the difference between replaying forty minutes and replaying seven days.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --execute


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --execute


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



GROUP                          TOPIC                          PARTITION  NEW-OFFSET     cg-user15-support              orders-user15                  0          0              cg-user15-support              orders-user15                  1          0              cg-user15-support              orders-user15                  2          0              
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\consume.bat --from-beginning --max-messages 20 --timeout-ms 30000
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\consume.bat --from-beginning --max-messages 25 --timeout-ms 30000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


order-1001
order-1002
recover-probe
day1-test-msg
order-notebook-test
recover-probe
recover-probe
recover-probe


Processed a total of 8 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo === AFTER reset ===
=== AFTER reset ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          0               1               1               -               -               -
cg-user15-support orders-user15   1          0               5               5               -               -               -
cg-user15-support orders-user15   2          0               2               2               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Diagnosis and close — Scenario 3

You have now done the replay, so you know exactly what it costs: every message on the topic was delivered a second time, and duplicates appeared in the consumer output.

**In this lab a duplicate prints a harmless line. In production it does whatever the consumer does** — inserts a row, sends an email, charges a card, ships an order. Kafka cannot tell the difference, and neither can the person who asked you to "just replay it".

### The update you write on the ticket

> Investigated INC-4502 before performing any replay.
>
> The consumer group's committed offset on partition 0 is already past the offset where `order-7700` was written, which means Kafka delivered that message to the consumer successfully. The gap is between the consumer and the database, not in Kafka.
>
> A replay would therefore not fix the missing record, and would redeliver every message on the topic for the replayed period — with a duplicate risk to downstream processing that I would need the application owner to accept in writing first.
>
> Recommended instead: the application team check their consumer logs around that offset for a processing or insert failure. If a targeted replay is still wanted afterwards, I can reset to a specific timestamp with `--to-datetime` rather than to the beginning, which limits the redelivery to the affected window. Happy to run the dry-run so we can review the exact scope before anything changes.

### What you would deliberately not do, and why

| Not this | Why not |
|----------|---------|
| Run the replay because the ticket asked for it | The evidence showed it would not fix the problem and would create a new one |
| Use `--to-earliest` for a forty-minute gap | It redelivers everything on the topic. `--to-datetime` targets only the affected window. |
| Execute without the dry-run | You lose your approval gate and your record of intent |
| Reset while a consumer is running | It will re-commit and undo the reset, leaving an inconsistent state |


---
## Scenario 4 — Security and configuration

### The ticket

> **INC-4515** — raised by the Inventory service team, 16:20
>
> *"Our producer started failing this afternoon with an authorization error. Nobody deployed anything. Can you reset the offsets to clear it?"*

### Spot the wrong fix in the request

The reporter has asked for an offset reset to solve an authorization error. Those two things are unrelated, and understanding why is the whole point of this scenario.

An offset reset changes **where a consumer group reads from**. An authorization error means **the broker refused an operation for this identity**. A reset cannot grant a permission, and this request is for a *producer* — which has no offsets at all.

Saying "that will not help, and here is what will" is a more useful answer than doing what was asked.

### Read the exception name — it tells you which gate stopped you

| Exception | Which gate | What it means | Who fixes it |
|-----------|-----------|---------------|--------------|
| `SaslAuthenticationException` | **Authentication** | Wrong username or password. You never got in. | Whoever holds the credentials |
| `UnsupportedSaslMechanismException` on port 9198 | **Listener mismatch** | A SCRAM client pointed at the IAM listener | Client configuration |
| `TopicAuthorizationException` | **Authorization** | You are authenticated, but not allowed on this topic | An ACL change |
| `GroupAuthorizationException` | **Authorization** | Authenticated, but not allowed to use this consumer group | An ACL change |
| `ClusterAuthorizationException` | **Authorization** | Authenticated, but not allowed to perform a cluster-wide operation such as editing ACLs | The platform team |

**Authentication happens first; authorization is checked afterwards, per operation.** So a `TopicAuthorizationException` actually tells you something positive as well: the credentials are correct and the network path works. Only the permission is missing. That narrows the investigation considerably.

### What "nobody deployed anything" usually means here

Authorization errors that appear without a deployment normally have one of three causes, none of which involve the application changing:

1. An ACL was added or removed by someone else — often for a different, legitimate reason
2. The client is now connecting on a different **listener** than before, so its authentication method no longer matches
3. Credentials were rotated somewhere, and one component still has the old ones

All three are checkable, and none of them are fixed by resetting offsets.


### What each cell in this scenario proves

| Cell | What it demonstrates |
|------|----------------------|
| SCRAM client against port **9198** | A deliberate failure. Proves a listener accepts only its own authentication mechanism. |
| SCRAM client against port **9196** | The correct pairing works — TLS and SCRAM are both fine from your machine |
| `kafka-acls --list` and produce a probe | What permissions currently exist, and whether produce is actually allowed right now |
| IAM client against port **9198** | The second authentication path, when the JAR and properties are in place |
| `kafka-configs --describe` | Whether a topic-level configuration override is involved |

**The deliberate failure is worth doing carefully.** Expect an error naming `SCRAM-SHA-512 not enabled`, or listing mechanisms `[OAUTHBEARER, AWS_MSK_IAM]`. Port 9198 expects IAM, so a SCRAM properties file is rejected before any permission is even considered.

That error is exactly what a real listener misconfiguration looks like — an application that worked yesterday failing to authenticate today because its endpoint changed. Recognising the message saves you from investigating credentials that were never the problem.

**For the IAM path on 9198 you need three things:** `%CLIENT_IAM%` present, the IAM JAR on `CLASSPATH`, and an IAM policy permitting MSK access. If IAM fails, record the error and move on — the successful SCRAM `--list` on 9196 already proves connectivity for this scenario.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo Wrong listener — expect failure:
kafka-topics.bat --bootstrap-server %BOOTSTRAP_IAM% --command-config %CLIENT% --list


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo Expect failure — SCRAM client on IAM port:
Expect failure — SCRAM client on IAM port:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-topics.bat --bootstrap-server %BOOTSTRAP_IAM% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Error while executing topic command : Client SASL mechanism 'SCRAM-SHA-512' not enabled in the server, enabled mechanisms are [OAUTHBEARER, AWS_MSK_IAM]

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo Correct SCRAM listener:
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


acl-lab-user15
orders-demo
orders-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-acls.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list --topic %ACL_TOPIC%
call ..\scripts\produce.bat %ACL_TOPIC% s4-probe


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-acls.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list --topic %ACL_TOPIC%


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Current ACLs for resource `ResourcePattern(resourceType=TOPIC, name=acl-lab-user15, patternType=LITERAL)`: 
 	(principal=User:user15, host=*, operation=DESCRIBE_CONFIGS, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=DESCRIBE, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=READ, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=WRITE, permissionType=ALLOW) 


c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\produce.bat %ACL_TOPIC% acl-ok


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
set CLASSPATH=C:\kafka\kafka_2.13-3.8.1\libs\aws-msk-iam-auth.jar;%CLASSPATH%
kafka-topics.bat --bootstrap-server %BOOTSTRAP_IAM% --command-config %CLIENT_IAM% --list
kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


acl-lab-user15
orders-demo
orders-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Dynamic configs for topic orders-user15 are:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Diagnosis and close — Scenario 4

**If `s4-probe` failed with an authorization error:** a deny ACL from Day 4 section 4a is probably still in place. That is a realistic finding in itself — a leftover rule from earlier work, blocking a client that used to function. Remove the deny, then send a fresh probe:

`call ..\scripts\produce.bat %ACL_TOPIC% s4-restored`

A successful send proves the authorization path is restored. This before-and-after pair is your evidence: produce failed, one permission changed, produce succeeded.

### The update you write on the ticket

> Investigated INC-4515. This is an authorization issue, not an offset issue — an offset reset would have had no effect, and a producer has no consumer offsets to reset.
>
> `TopicAuthorizationException` means the client authenticated successfully and was then refused permission for that specific topic, so the credentials and the network path are both fine. `kafka-acls --list` showed a deny rule on the topic, which was blocking Write for the service principal.
>
> The rule has been removed and produce validated with a test message at 16:34. Please confirm your service has recovered. For the root cause I would like to establish who added that rule and why, so it is not reintroduced.

### What you would deliberately not do, and why

| Not this | Why not |
|----------|---------|
| Reset offsets as requested | Unrelated to authorization, and it would have created duplicate processing for nothing |
| Grant broad ALLOW permissions to make the error stop | It fixes the symptom by removing the control. Change the specific rule that is wrong. |
| Assume the credentials were wrong | `TopicAuthorizationException` proves they were right. Reading the exception name saved that whole investigation. |


---
## Scenario 5 — End-to-end incident

### The ticket

> **INC-4530** — raised by the on-call duty manager, 16:15
>
> *"Multiple order problems reported this afternoon. Publishing is failing, the dashboard is behind, and at least one order is missing from the database. Application log attached. We need a root cause by end of day."*

### Why this one is different

The previous four scenarios each had one problem. This log has **four separate symptoms**, arriving over seven minutes, and they are not all the same incident.

This is what real logs look like, and the skill being tested is not command knowledge. It is the discipline to separate the symptoms, decide which one is the cause and which are consequences, and pick **one** primary fix rather than firing at all four.

### The rules for this scenario

| Rule | Why |
|------|-----|
| Read the **whole** log before acting | The first error is rarely the root cause; it is usually just the earliest symptom |
| Choose **one** primary fix | Four changes at once means you never learn which mattered, and cannot write an honest root cause |
| Label every fact `[log]` or `[live]` | Somebody reading your report has to know which facts are current |
| Validate with `s5-validate` | A named test message plus `LAG` 0 is proof. "It looks better" is not. |


In [ ]:
%%cmd
type ..\day-05\samples\scenario-5-app.log


2026-08-20 16:02:01,009 INFO  com.example.orders.OrderProducer - starting producer topic=orders-lab acks=all
2026-08-20 16:02:18,441 WARN  org.apache.kafka.clients.producer.internals.Sender - [Producer clientId=order-producer-lab] Got error produce response with correlation id 902 on topic-partition orders-lab-0, retrying (remaining retries 3). Error: NOT_LEADER_OR_FOLLOWER
2026-08-20 16:04:18,880 ERROR com.example.orders.OrderProducer - publish failed key=order-8801
org.apache.kafka.common.errors.TimeoutException: Expiring 1 record(s) for orders-lab-0:120000 ms has passed since batch creation

2026-08-20 16:05:02,100 INFO  com.example.orders.OrderConsumer - poll loop group=cg-lab-support
2026-08-20 16:05:40,212 WARN  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=cg-lab-support] consumer poll timeout has expired.
2026-08-20 16:05:40,440 ERROR org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - Offset commi

### Read the log as a timeline

Work through it in order and separate the four symptoms:

| Time | What the log says | What it means | Symptom or cause? |
|------|-------------------|---------------|-------------------|
| 16:02:18 | `NOT_LEADER_OR_FOLLOWER` on `orders-lab-0`, retrying | The partition leader moved. The client retries automatically. | Normal in isolation — but a clue |
| 16:04:18 | `TimeoutException` — `publish failed key=order-8801` | After 120 seconds of retries the message was discarded and **never reached the topic** | Symptom |
| 16:05:40 | `poll timeout has expired`, then `CommitFailedException` | The consumer was too slow, was evicted from the group, and could not save its progress | A **separate** problem from the producer's |
| 16:08:11 | `Missing order-7700 in downstream DB; last committed offset on partition 0 is 9100` | The consumer had **already committed past** this message | Decisive |
| 16:09:44 | `Connection to node 1 ... could not be established` | Broker 1 was unreachable from the application host | Likely the underlying cause |

### Now separate the two orders being discussed

This is the trap in the ticket, and getting it right is most of the exercise:

| Order | What the log shows | Can a replay recover it? |
|-------|--------------------|--------------------------|
| **`order-8801`** | Publish **failed** with a timeout. It never reached the topic. | **No.** Kafka never received it. It must be re-sent by the producing application. |
| **`order-7700`** | Committed offset is already **past** it, so it was delivered | **No.** Kafka delivered it; the consumer failed to write it to the database. |

**Neither missing order is recoverable by replaying Kafka.** One never arrived; the other arrived and was mishandled downstream. That single conclusion is worth more than any command you could run, and it is the finding that keeps you from causing a second incident.

Note the application's own line at 16:08:11 — `will not reset offsets automatically`. Even the application was written to refuse this shortcut.

### Decision tree — pick one primary fix

| Evidence in the log | The primary fix |
|---------------------|-----------------|
| Broker unreachable, producer timeouts, ISR healthy on the live check | Escalate the broker or network path with your metrics; validate with a probe |
| Poll timeout and `CommitFailedException` | Restore the consumer path |
| `TopicAuthorizationException` | Remove the blocking ACL |
| A message missing but still within retention **and** not yet committed past | A targeted reset, dry-run first |

For this log, the broker connectivity failure at 16:09:44 is the item that best explains the producer timeouts and the leader change. The consumer eviction is a second, independent problem worth its own ticket rather than being folded into this one.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === Evidence snapshot ===
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000
kafka-acls.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list --topic %ACL_TOPIC%


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo === BEFORE reset ===
=== BEFORE reset ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-acls.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list --topic %ACL_TOPIC%


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Current ACLs for resource `ResourcePattern(resourceType=TOPIC, name=acl-lab-user15, patternType=LITERAL)`: 
 	(principal=User:user15, host=*, operation=DESCRIBE_CONFIGS, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=DESCRIBE, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=READ, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=WRITE, permissionType=ALLOW) 


c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Validate the restore

`s5-validate` is the same technique you have used all week: publish a message whose text you chose, consume it, and confirm `LAG` returns to 0.

**What each part of the next cell proves:**

| Part | What it establishes |
|------|---------------------|
| `produce.bat s5-validate` succeeds | A new message can be written to the topic **now** |
| `s5-validate` appears in the consumer output | The full path — producer, broker, consumer — is working |
| Final `--describe` shows `LAG` 0 | The group is caught up, with no backlog left |

All three together are what lets you resolve the ticket with evidence rather than an opinion.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat s5-validate
call ..\scripts\consume.bat --max-messages 10 --timeout-ms 25000
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\produce.bat recover-probe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\consume.bat --from-beginning --max-messages 15 --timeout-ms 25000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


...

Processed a total of 1 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>echo === AFTER recover — LAG should be 0 or lower ===
=== AFTER recover — LAG should be 0 or lower ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### The live health check — run it, do not just read it

A scenario is only closed when **live commands** confirm health. The next cell runs three checks in order:

| # | Check | What a pass looks like |
|---|-------|------------------------|
| 1 | Cluster state | `ACTIVE`. An `AccessDenied` here means your AWS CLI is in a different account — note it and rely on the Kafka checks. |
| 2 | Topic `--describe` | Replication factor 3, with `Isr` matching `Replicas` on every partition |
| 3 | Group `--describe` | `LAG` at 0 after your restore action |

Together these prove the data plane is healthy from an independent client. That combination is the evidence you attach when you resolve the ticket, and it is what makes the resolution defensible if the problem recurs.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION% --query "ClusterInfo.State" --output text
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


### Write the incident summary — Scenario 5

This is the deliverable. Use the structure below; it is the structure a duty manager expects, and it works because it separates what happened from what you did about it.

**1. Summary — two sentences, no jargon**

> Between 16:02 and 16:10 the order service could not reliably publish to Kafka and its consumer fell out of its consumer group. One order was not published at all and a second was published but not written to the database.

**2. Timeline** — with every entry labelled

> `[log]` 16:02:18 — partition leader change on `orders-lab-0`, client retried
> `[log]` 16:04:18 — publish of `order-8801` failed after 120s; message never reached the topic
> `[log]` 16:05:40 — consumer breached its poll interval, was evicted, offset commit failed
> `[log]` 16:08:11 — `order-7700` missing downstream; committed offset already past it
> `[log]` 16:09:44 — broker 1 unreachable from the application host
> `[live]` 16:40 — cluster ACTIVE, ISR full on all partitions, test message published and consumed, lag 0

**3. Root cause** — one paragraph, and be honest about the limits

> Loss of connectivity from the application host to broker 1 caused the leader change and the publish timeouts. The consumer eviction in the same window appears to be an independent problem with processing time rather than a consequence of the connectivity loss; it needs its own investigation.

**4. Impact** — in business terms, not technical ones

> `order-8801` was never published and must be re-submitted by the order service. `order-7700` was delivered by Kafka but not written to the database, so it requires a targeted fix downstream rather than a Kafka replay. Messages between the last successful commit and the eviction will have been reprocessed on restart.

**5. Actions taken, and actions not taken**

> Restored and validated the consumer path with a test message; lag returned to 0. I did **not** replay from Kafka: neither missing order is recoverable that way, and a replay would have redelivered every other message in the window with a real duplicate risk downstream.

**6. Follow-up**

> Application team to re-submit `order-8801` and investigate the `order-7700` insert failure. Network team to confirm the connectivity loss to broker 1 at 16:09. Separate ticket for the consumer poll-interval breach.

**Point 5 is where experience shows.** Anyone can list what they changed. Stating clearly what you chose *not* to do, and why, is what tells a reader that the incident was handled rather than merely survived.


---
## Best practices — the daily operations checklist

Everything so far has been reactive: a ticket arrives and you respond. This is the other half of the job, and the half that reduces the number of tickets.

Run through [samples/ops-checklist.md](samples/ops-checklist.md) once now, so you know what a normal cluster looks like.

### Why a daily walkthrough is worth the ten minutes

| Reason | What it gives you |
|--------|-------------------|
| You learn the **baseline** | "CPU is at 60%" only means something if you know it is normally 20% |
| You catch **trends** before they become incidents | Disk at 70% and climbing is a planned change. Disk at 100% is an outage. |
| You find **leftovers** | A deny ACL from last week's work, still blocking a client |
| You keep the **evidence habit** warm | The commands are already familiar when you actually need them at 2 a.m. |

### The three habits worth taking away from this week

**1. Evidence before change.** Every command in this course either gathers evidence or validates a change. Very few of them change anything. That ratio is not an accident — it is what the job actually looks like.

**2. One change at a time.** Then validate. If it did not help, you have ruled something out, which is progress you can report.

**3. Validate with a named test message.** Not "the process is running". A message you published, arriving where it should, at a time you can point to.


In [ ]:
%%cmd
type ..\day-05\samples\ops-checklist.md


# Daily Kafka / MSK operations checklist

Cluster: `<cluster-name>`  
Client: Windows 10 lab VM (**outside MSK VPC**; public SCRAM **9196**)  
Auth default: SASL/SCRAM (`%USERPROFILE%\client-scram.properties`; username = login `userN`)  
Topic / group: `orders-userN` / `cg-userN-support`

| Check | Command or place | OK? |
|-------|------------------|-----|
| Cluster state ACTIVE | `aws kafka describe-cluster` | |
| Broker count matches expectation | same | |
| Public SCRAM bootstrap resolves | `get-bootstrap-brokers` / `nslookup` on `-public` host | |
| Off-VPC Windows `--list` works | `kafka-topics.bat --list` on **9196** | |
| Critical topics ISR = RF | `--describe` | |
| Consumer members present | `--describe --group` | |
| Lag not trending up | group describe / dashboard | |
| CloudWatch alarms | `describe-alarms` (lab alarm prefix); empty list â†’ mark N/A, do not create | |
| CpuIdle not exhausted on all brokers | CloudWatch dashboard / metrics | |
| Kafka log disk under thresho

## Assignment / review

Pick **one** scenario and write it up in full, as though handing it to a colleague who was not there.

| Section | What it must contain |
|---------|----------------------|
| **Symptom** | What the ticket claimed, in the reporter's words |
| **Scope** | One topic and group, or cluster-wide — and how you established that |
| **Evidence** | At least three facts, each labelled `[log]` or `[live]`, with timestamps |
| **Diagnosis** | Which layer was at fault: application, consumer, broker, or network path |
| **The one fix** | What you changed, and why that one and not the others |
| **Validation** | The named test message and the `LAG` value that proved it worked |
| **Not done** | At least one action you deliberately avoided, with the reason |

**The two questions you should be ready to answer out loud:**

1. *How did you know it was not a Kafka problem?* — or, if it was, how did you rule out the client side first?
2. *What would you have broken if you had done the obvious thing?*

The second one is the point of the whole week. On every one of these five scenarios, the obvious action was available, quick, and wrong.


## Wrap-up

Over five days you have gone from `--list` to running a full incident with evidence, a single controlled fix, and a written root cause.

### What you can now do

| Skill | Where you built it |
|-------|--------------------|
| Connect to MSK over TLS with SCRAM, from outside the VPC | Day 1 |
| Read partitions, leaders, ISR, offsets and lag from `--describe` | Days 1 and 2 |
| Diagnose an incident from application logs, line by line | Day 2 |
| Read broker metrics and logs in CloudWatch, and set alarms that people will trust | Day 3 |
| Reset offsets and replay safely, understanding what it costs downstream | Day 4 |
| Separate authentication from authorization by reading the exception name | Day 4 |
| Run a ticket end to end: evidence, one fix, validation, written root cause | Day 5 |

### The one thing to remember

Most Kafka tickets are not Kafka problems. They are consumer problems, client configuration problems, or network problems, reported as Kafka problems because that is where the data was last seen.

The skill that makes you useful is not knowing more commands. It is the twenty minutes of evidence-gathering that tells you which layer to fix — and the confidence to say "the cluster is healthy, here is the proof, the problem is elsewhere".

Be ready to explain **one** scenario end to end: symptom, evidence, fix, validation, and what you deliberately did **not** do.
